In [2]:
# =========================================================
# NOTEBOOK 07 — Feature Engineering v2 (Experimento)
# =========================================================
# REGLA DE ORO: este notebook es de SOLO LECTURA respecto
# a todos los artefactos existentes. No modifica ni
# sobreescribe ningún archivo de data/processed/ ni models/
#
# Flujo:
#   1. Carga artefactos existentes (read-only)
#   2. Construye nuevas features sobre train y test
#   3. Evalúa con mismo CV, mismos pesos, mismo modelo tuned
#   4. Compara contra baseline 0.4871
#   5. Exporta SOLO si mejora > 0.003 y SOLO a rutas nuevas
# =========================================================

from triaje_ia.config import DATA_PROCESSED, MODELS_DIR, FIGURES_DIR
from triaje_ia.data.cleaner import cargar_dataset_limpio
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, classification_report
import lightgbm as lgb
from lightgbm import LGBMClassifier
import pandas as pd
import numpy as np
import json
import warnings

warnings.filterwarnings("ignore")

def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro", zero_division=0)

# ---------------------------------------------------------
# 1. CARGA DE ARTEFACTOS EXISTENTES — read-only
# ---------------------------------------------------------
X_train = pd.read_parquet(DATA_PROCESSED / "X_train.parquet")
X_test  = pd.read_parquet(DATA_PROCESSED / "X_test.parquet")
y_train = pd.read_parquet(DATA_PROCESSED / "y_train.parquet").squeeze()
y_test  = pd.read_parquet(DATA_PROCESSED / "y_test.parquet").squeeze()

groups_train         = np.load(DATA_PROCESSED / "groups_train.npy")
sample_weights_train = np.load(DATA_PROCESSED / "sample_weights_train.npy")  # sqrt-balanced ya guardado

with open(DATA_PROCESSED / "feature_config_selected.json", encoding="utf-8") as f:
    feature_config = json.load(f)

FEATURES_SELECCIONADAS = feature_config["features_seleccionadas"]            # 46 features del pipeline original
TARGET                 = feature_config["target"]

X_tr_sel = X_train[FEATURES_SELECCIONADAS].copy()                            # subconjunto limpio — base del experimento
X_te_sel = X_test[FEATURES_SELECCIONADAS].copy()

print(f"Artefactos cargados — X_train: {X_train.shape} | X_tr_sel: {X_tr_sel.shape}")

# ---------------------------------------------------------
# 2. CARGA DEL DATAFRAME ORIGINAL
# ---------------------------------------------------------
# Necesitamos columnas que no están en las 46 seleccionadas:
# - llegada_autonoma: para perfil_no_urgente y cronico_estable
# - pain continuo: ya está en X_tr_sel ✅
# - age continuo: ya está en X_tr_sel ✅
# ---------------------------------------------------------
df_original = cargar_dataset_limpio(forzar=False)

# Recuperar el mismo split temporal que en el pipeline original
fecha_corte = df_original["intime"].quantile(0.80)
mask_train  = df_original["intime"] <= fecha_corte
mask_test   = df_original["intime"] >  fecha_corte

df_train = df_original[mask_train].reset_index(drop=True)
df_test  = df_original[mask_test].reset_index(drop=True)

assert len(df_train) == len(X_train), \
    f"Desalineación train: df={len(df_train)}, X={len(X_train)}"
assert len(df_test) == len(X_test), \
    f"Desalineación test: df={len(df_test)}, X={len(X_test)}"

print(f"df_train: {df_train.shape} | df_test: {df_test.shape} — alineación ✅")

# ---------------------------------------------------------
# 3. CONSTRUCCIÓN DE NUEVAS FEATURES
# ---------------------------------------------------------
# Todas las funciones reciben (df_split, X_split) para poder
# usar tanto columnas del df original como features ya
# engineerizadas que están en X_train.
# ---------------------------------------------------------

def construir_features_v2(df, X):
    feats = pd.DataFrame(index=df.index)

    # Extraer columnas con posibles NaN una sola vez, ya imputadas
    pain          = X["pain"].fillna(0)
    age           = X["age"].fillna(0)
    n_vit         = X["n_vitales_anomalos"].fillna(0)
    n_med         = X["n_medicamentos"].fillna(0)
    zona_verde    = X["zona_verde"].fillna(0)
    sin_med       = X["sin_medicacion"].fillna(0)
    o2sat_bajo    = X["o2sat_bajo_92"].fillna(0)
    news2_alto    = X["news2_alto"].fillna(0)
    shock_alto    = X["shock_index_alto"].fillna(0)
    cc_dolor      = X["cc_dolor_toracico"].fillna(0)
    cc_disnea     = X["cc_disnea"].fillna(0)
    cc_neuro      = X["cc_neuro_ams"].fillna(0)
    cc_trauma     = X["cc_trauma"].fillna(0)
    med_opiaceo   = X["med_opiaceo"].fillna(0)
    alto_sangrado = X["alto_riesgo_sangrado"].fillna(0)

    llegada_autonoma = df["arrival_transport"].isin(
        ["WALK IN", "SELF", "AMBULATORY"]
    ).astype("int8")

    # ---- BLOQUE A: Perfil bajo riesgo ----
    feats["perfil_no_urgente"] = (
        (zona_verde    == 1) &
        (sin_med       == 1) &
        (llegada_autonoma == 1) &
        (pain          <= 3) &
        (n_vit         == 0)
    ).astype("int8")

    feats["cronico_estable"] = (
        (n_med         >= 5) &
        (zona_verde    == 1) &
        (llegada_autonoma == 1)
    ).astype("int8")

    # ---- BLOQUE B: Interacciones CC × Vitales ----
    feats["dolor_toracico_critico"] = (
        (cc_dolor   == 1) &
        (shock_alto == 1)
    ).astype("int8")

    feats["dolor_toracico_estable"] = (
        (cc_dolor   == 1) &
        (zona_verde == 1)
    ).astype("int8")

    feats["disnea_hipoxia"] = (
        (cc_disnea   == 1) &
        (o2sat_bajo  == 1)
    ).astype("int8")

    feats["disnea_compensada"] = (
        (cc_disnea   == 1) &
        (o2sat_bajo  == 0) &
        (zona_verde  == 1)
    ).astype("int8")

    # ---- BLOQUE C: Interacciones edad × CC ----
    feats["trauma_anciano"] = (
        (cc_trauma == 1) &
        (age       >= 65)
    ).astype("int8")

    feats["neuro_anciano"] = (
        (cc_neuro == 1) &
        (age      >= 65)
    ).astype("int8")

    feats["trauma_anticoagulado"] = (
        (cc_trauma    == 1) &
        (alto_sangrado == 1)
    ).astype("int8")

    # ---- BLOQUE D: Señales ordinales pain/NEWS2 ----
    feats["pain_sin_deterioro"] = (
        (pain       >= 7) &
        (news2_alto == 0)
    ).astype("int8")

    feats["deterioro_sin_dolor"] = (
        (pain       <= 3) &
        (news2_alto == 1)
    ).astype("int8")

    # ---- BLOQUE E: Contexto farmacológico ----
    feats["dolor_enmascarado_opiaceo"] = (
        (med_opiaceo == 1) &
        (pain        <= 3)
    ).astype("int8")

    # ---- BLOQUE F: Complejidad clínica ----
    cc_cols = [c for c in FEATURES_SELECCIONADAS if c.startswith("cc_")]
    feats["n_cc_activos"] = X[cc_cols].fillna(0).sum(axis=1).astype("int8")

    feats["alarma_sin_expresion"] = (
        ((cc_dolor == 1) | (cc_disnea == 1) | (cc_neuro == 1)) &
        (zona_verde == 1) &
        (n_vit      == 0)
    ).astype("int8")

    return feats


# Construir sobre train y test
print("\nConstruyendo features v2...")
feats_train = construir_features_v2(df_train, X_tr_sel)
feats_test  = construir_features_v2(df_test,  X_te_sel)

FEATURES_NUEVAS = list(feats_train.columns)
print(f"Features nuevas construidas: {len(FEATURES_NUEVAS)}")
print(f"  {FEATURES_NUEVAS}")

# Concatenar con features existentes — SIN modificar X_tr_sel original
X_tr_v2 = pd.concat([X_tr_sel.reset_index(drop=True),
                      feats_train.reset_index(drop=True)], axis=1)
X_te_v2 = pd.concat([X_te_sel.reset_index(drop=True),
                      feats_test.reset_index(drop=True)],  axis=1)

print(f"\nX_tr_v2: {X_tr_v2.shape}  (46 originales + {len(FEATURES_NUEVAS)} nuevas)")

# ---------------------------------------------------------
# 4. VALIDACIÓN ESTADÍSTICA RÁPIDA DE LAS NUEVAS FEATURES
# ---------------------------------------------------------
from scipy.stats import kruskal

print("\n" + "=" * 65)
print("  VALIDACIÓN UNIVARIANTE — nuevas features vs acuity")
print("=" * 65)
print(f"  {'Feature':<30} {'Prevalencia':>12} {'H (Kruskal)':>13}")
print("-" * 58)

features_validas = []
for col in FEATURES_NUEVAS:
    prevalencia = feats_train[col].mean()
    grupos      = [feats_train[col][y_train == k].values for k in sorted(y_train.unique())]
    # Solo testear si hay varianza suficiente
    if feats_train[col].std() > 0 and prevalencia > 0.001:
        H, p = kruskal(*grupos)
        flag = "✅" if H > 100 else ("⚠️ " if H > 20 else "🚨")
        print(f"  {flag} {col:<28} {prevalencia:>12.3f} {H:>13.1f}")
        if H > 20:                                                            # umbral mínimo de señal univariante
            features_validas.append(col)
    else:
        print(f"  🚨 {col:<28} {prevalencia:>12.3f} {'sin varianza':>13}")

print(f"\n  Features con señal suficiente (H>20): {len(features_validas)} de {len(FEATURES_NUEVAS)}")

# ---------------------------------------------------------
# 5. CV CON FEATURES ORIGINALES + NUEVAS VÁLIDAS
# ---------------------------------------------------------
# Usamos solo las features que pasaron el filtro univariante
# para no añadir ruido al modelo
# ---------------------------------------------------------
FEATURES_V2 = FEATURES_SELECCIONADAS + features_validas                      # 46 originales + nuevas validadas

X_tr_v2_filtrado = X_tr_v2[FEATURES_V2]

# Cargar mejores params del tuning
with open(MODELS_DIR / "lgbm_best_params.json", encoding="utf-8") as f:
    best_params = json.load(f)

params_modelo = {k: v for k, v in best_params.items()
                 if not k.startswith("_")}                                    # eliminar metadata antes de pasar al modelo

CV = StratifiedGroupKFold(n_splits=5, shuffle=False)

lgbm_v2 = LGBMClassifier(**params_modelo)

scores_v2  = []
n_trees_v2 = []

print("\n" + "=" * 65)
print(f"  CV — LightGBM tuned con {len(FEATURES_V2)} features ({len(features_validas)} nuevas)")
print("=" * 65)

for fold, (idx_tr, idx_val) in enumerate(CV.split(X_tr_v2_filtrado, y_train, groups=groups_train)):

    X_fold_tr  = X_tr_v2_filtrado.iloc[idx_tr]
    X_fold_val = X_tr_v2_filtrado.iloc[idx_val]
    y_fold_tr  = y_train.iloc[idx_tr] - 1
    y_fold_val = y_train.iloc[idx_val] - 1
    sw_fold_tr  = sample_weights_train[idx_tr]
    sw_fold_val = sample_weights_train[idx_val]

    lgbm_v2.fit(
        X_fold_tr, y_fold_tr,
        sample_weight      = sw_fold_tr,
        eval_set           = [(X_fold_val, y_fold_val)],
        eval_sample_weight = [sw_fold_val],
        callbacks          = [
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(period=-1),
        ],
    )

    y_pred = lgbm_v2.predict(X_fold_val) + 1
    score  = macro_f1(y_train.iloc[idx_val], y_pred)
    scores_v2.append(score)
    n_trees_v2.append(lgbm_v2.best_iteration_)
    print(f"  Fold {fold+1} — Macro F1: {score:.4f}  |  árboles: {lgbm_v2.best_iteration_}")

f1_v2       = np.mean(scores_v2)
f1_baseline = 0.4871                                                          # mejor resultado del tuning anterior
ganancia    = f1_v2 - f1_baseline

print(f"\n  LightGBM v2 ({len(FEATURES_V2)} features): {f1_v2:.4f} ± {np.std(scores_v2):.4f}")
print(f"  Baseline tuned (46 features):          {f1_baseline:.4f}")
print(f"  Ganancia features nuevas:             {ganancia:+.4f}")
print(f"  Árboles promedio: {np.mean(n_trees_v2):.0f}")

# ---------------------------------------------------------
# 6. CLASSIFICATION REPORT COMPARATIVO
# ---------------------------------------------------------
from sklearn.metrics import precision_recall_fscore_support

oof_v2 = np.zeros(len(y_train), dtype=np.int8)

for fold, (idx_tr, idx_val) in enumerate(CV.split(X_tr_v2_filtrado, y_train, groups=groups_train)):
    X_fold_tr  = X_tr_v2_filtrado.iloc[idx_tr]
    X_fold_val = X_tr_v2_filtrado.iloc[idx_val]
    y_fold_tr  = y_train.iloc[idx_tr] - 1
    y_fold_val = y_train.iloc[idx_val] - 1
    sw_fold_tr  = sample_weights_train[idx_tr]
    sw_fold_val = sample_weights_train[idx_val]

    lgbm_v2.fit(
        X_fold_tr, y_fold_tr,
        sample_weight      = sw_fold_tr,
        eval_set           = [(X_fold_val, y_fold_val)],
        eval_sample_weight = [sw_fold_val],
        callbacks          = [
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(period=-1),
        ],
    )
    oof_v2[idx_val] = (lgbm_v2.predict(X_fold_val) + 1).astype(np.int8)

print("\n" + "=" * 65)
print("  COMPARATIVA POR CLASE — baseline vs v2")
print("=" * 65)

# Cargar preds OOF del baseline para comparar
# (asumimos que oof_preds_xgb o oof_preds_sqrt están en memoria)
# Si no están, recalculamos solo el report de v2
try:
    p_b, r_b, f_b, _ = precision_recall_fscore_support(
        y_train, oof_preds_sqrt, zero_division=0                             # oof del baseline sqrt — debe estar en memoria del nb anterior
    )
    tiene_baseline_oof = True
except NameError:
    tiene_baseline_oof = False
    print("  (oof_preds_sqrt no en memoria — mostrando solo v2)")

p_v, r_v, f_v, _ = precision_recall_fscore_support(y_train, oof_v2, zero_division=0)
nombres = ["1-Crítico", "2-Emergencia", "3-Urgente", "4-Menos urg.", "5-No urg."]

if tiene_baseline_oof:
    print(f"\n  {'Clase':<16} {'F1 base':>8} {'F1 v2':>8} {'Δ F1':>8}  {'R base':>7} {'R v2':>7}")
    print("-" * 60)
    for i in range(5):
        delta = f_v[i] - f_b[i]
        flag  = "🔺" if delta > 0.01 else ("🔻" if delta < -0.01 else "  ")
        print(f"  {nombres[i]:<14} {f_b[i]:>8.3f} {f_v[i]:>8.3f} {flag}{delta:>+6.3f}  "
              f"{r_b[i]:>7.3f} {r_v[i]:>7.3f}")
else:
    print(f"\n  {'Clase':<16} {'F1 v2':>8} {'Recall':>8} {'Precision':>10}")
    print("-" * 48)
    for i in range(5):
        print(f"  {nombres[i]:<14} {f_v[i]:>8.3f} {r_v[i]:>8.3f} {p_v[i]:>10.3f}")

# ---------------------------------------------------------
# 7. DECISIÓN DE EXPORTACIÓN
# ---------------------------------------------------------
UMBRAL_GANANCIA = 0.003                                                       # ganancia mínima para justificar complejidad añadida

print("\n" + "=" * 65)
print("  DECISIÓN")
print("=" * 65)

if ganancia > UMBRAL_GANANCIA:
    print(f"  ✅ Ganancia {ganancia:+.4f} > umbral {UMBRAL_GANANCIA}")
    print(f"  → Features v2 INCORPORABLES al pipeline")
    print(f"  → Exportar a DATA_PROCESSED como X_train_v2.parquet")

    # Exportar SOLO a rutas nuevas — nunca sobreescribir las originales
    X_tr_v2_filtrado.to_parquet(
        DATA_PROCESSED / "X_train_v2.parquet", index=False
    )
    X_te_v2[FEATURES_V2].to_parquet(
        DATA_PROCESSED / "X_test_v2.parquet", index=False
    )

    config_v2 = {
        "features_originales":  FEATURES_SELECCIONADAS,
        "features_nuevas":      features_validas,
        "todas_features_v2":    FEATURES_V2,
        "n_features_v2":        len(FEATURES_V2),
        "macro_f1_baseline":    f1_baseline,
        "macro_f1_v2":          round(f1_v2, 6),
        "ganancia":             round(ganancia, 6),
    }
    with open(DATA_PROCESSED / "feature_config_v2.json", "w", encoding="utf-8") as f:
        json.dump(config_v2, f, indent=2, ensure_ascii=False)

    print(f"  Exportados: X_train_v2.parquet, X_test_v2.parquet, feature_config_v2.json")

else:
    print(f"  ❌ Ganancia {ganancia:+.4f} ≤ umbral {UMBRAL_GANANCIA}")
    print(f"  → Features v2 NO justifican complejidad añadida")
    print(f"  → Pipeline original (0.4871) se mantiene como definitivo")
    print(f"  → Ningún artefacto existente ha sido modificado ✅")

print("=" * 65)

2026-05-10 16:00:28.907 | WARNING  | triaje_ia.data.cleaner:cargar_dataset_limpio:193 - Cargando desde caché: C:\Users\CARLOS\triaje-ia-tfg\data\interim\dataset_clean.parquet. Si cambiaste el pipeline, usa forzar=True.


Artefactos cargados — X_train: (334480, 46) | X_tr_sel: (334480, 46)


2026-05-10 16:00:29.453 | SUCCESS  | triaje_ia.data.cleaner:cargar_dataset_limpio:198 - Cargado: 418,100 filas | 21 columnas


df_train: (334480, 21) | df_test: (83620, 21) — alineación ✅

Construyendo features v2...
Features nuevas construidas: 14
  ['perfil_no_urgente', 'cronico_estable', 'dolor_toracico_critico', 'dolor_toracico_estable', 'disnea_hipoxia', 'disnea_compensada', 'trauma_anciano', 'neuro_anciano', 'trauma_anticoagulado', 'pain_sin_deterioro', 'deterioro_sin_dolor', 'dolor_enmascarado_opiaceo', 'n_cc_activos', 'alarma_sin_expresion']

X_tr_v2: (334480, 60)  (46 originales + 14 nuevas)

  VALIDACIÓN UNIVARIANTE — nuevas features vs acuity
  Feature                         Prevalencia   H (Kruskal)
----------------------------------------------------------
  ✅ perfil_no_urgente                   0.028        4775.9
  ✅ cronico_estable                     0.078        1733.5
  ✅ dolor_toracico_critico              0.002         856.6
  ✅ dolor_toracico_estable              0.026        2598.7
  ✅ disnea_hipoxia                      0.003        6667.6
  ✅ disnea_compensada                   0.013 

In [3]:
# =========================================================
# EXPERIMENTO 2 — ccs_category + temperature
# =========================================================
# Dos señales disponibles en el dataset original que no
# están en las 46 features seleccionadas:
#   1. ccs_category → target encoding por acuity medio
#   2. temperature  → valor real (no solo el flag de missing)
# =========================================================

# ---------------------------------------------------------
# FEATURE 1: ccs_category — target encoding
# ---------------------------------------------------------
# Target encoding: sustituye cada categoría CCS por la
# media del acuity histórico de esa categoría.
# Se calcula SOLO sobre train para evitar leakage.
# Las categorías no vistas en train → mediana global.
# ---------------------------------------------------------

ccs_train = df_train["ccs_category"]
ccs_test  = df_test["ccs_category"]

# Media de acuity por categoría CCS — solo sobre train
ccs_mean_acuity = (
    y_train
    .groupby(ccs_train.values)
    .mean()
    .rename("ccs_mean_acuity")
)

mediana_global = y_train.median()                                            # fallback para categorías no vistas en train

feats_ccs_train = (
    ccs_train
    .map(ccs_mean_acuity)
    .fillna(mediana_global)
    .rename("ccs_mean_acuity")
    .reset_index(drop=True)
)

feats_ccs_test = (
    ccs_test
    .map(ccs_mean_acuity)
    .fillna(mediana_global)
    .rename("ccs_mean_acuity")
    .reset_index(drop=True)
)

print(f"ccs_mean_acuity — categorías únicas: {ccs_train.nunique()}")
print(f"  Rango train: [{feats_ccs_train.min():.3f}, {feats_ccs_train.max():.3f}]")
print(f"  Nulos train: {feats_ccs_train.isna().sum()} | test: {feats_ccs_test.isna().sum()}")

# Validación rápida — correlación con acuity
from scipy.stats import spearmanr
rho, p = spearmanr(feats_ccs_train, y_train)
print(f"  Spearman vs acuity: ρ={rho:.4f} (p={p:.2e})")                    # esperamos ρ > 0.3 para considerar la feature útil

# ---------------------------------------------------------
# FEATURE 2: temperature — valor real
# ---------------------------------------------------------
# temperature_missing ya está en las 46 features (flag de
# si se midió o no). Pero el valor real captura fiebre alta
# e hipotermia — señales de clase 1-2 que el flag no ve.
# Se imputa con mediana de train para los NaN.
# ---------------------------------------------------------

temp_mediana = df_train["temperature"].median()                              # mediana calculada solo sobre train — sin leakage

feats_temp_train = (
    df_train["temperature"]
    .fillna(temp_mediana)
    .rename("temperature_valor")
    .reset_index(drop=True)
)

feats_temp_test = (
    df_test["temperature"]
    .fillna(temp_mediana)
    .rename("temperature_valor")
    .reset_index(drop=True)
)

print(f"\ntemperature_valor:")
print(f"  Mediana imputación: {temp_mediana:.1f}°F")
print(f"  Nulos originales train: {df_train['temperature'].isna().sum():,} "
      f"({df_train['temperature'].isna().mean()*100:.1f}%)")

# Kruskal para validar señal
from scipy.stats import kruskal
grupos_temp = [feats_temp_train[y_train.values == k].values
               for k in sorted(y_train.unique())]
H_temp, _ = kruskal(*grupos_temp)

grupos_ccs  = [feats_ccs_train[y_train.values == k].values
               for k in sorted(y_train.unique())]
H_ccs, _   = kruskal(*grupos_ccs)

print(f"  H Kruskal vs acuity: {H_temp:.1f}")                               # comparar con las 14 features nuevas del experimento anterior
print(f"\nccs_mean_acuity H Kruskal: {H_ccs:.1f}")

ccs_mean_acuity — categorías únicas: 243
  Rango train: [1.000, 4.207]
  Nulos train: 0 | test: 0
  Spearman vs acuity: ρ=0.4975 (p=0.00e+00)

temperature_valor:
  Mediana imputación: 98.0°F
  Nulos originales train: 14,151 (4.2%)
  H Kruskal vs acuity: 405.6

ccs_mean_acuity H Kruskal: 84398.4


In [4]:
# ---------------------------------------------------------
# CONSTRUIR X AMPLIADO — 46 originales + ccs + temperature
# ---------------------------------------------------------
X_tr_exp2 = pd.concat([
    X_tr_sel.reset_index(drop=True),
    feats_ccs_train,
    feats_temp_train,
], axis=1)

X_te_exp2 = pd.concat([
    X_te_sel.reset_index(drop=True),
    feats_ccs_test,
    feats_temp_test,
], axis=1)

print(f"X_tr_exp2: {X_tr_exp2.shape}  (46 originales + ccs_mean_acuity + temperature_valor)")

# ---------------------------------------------------------
# CV — mismo modelo tuned, mismos pesos
# ---------------------------------------------------------
lgbm_exp2 = LGBMClassifier(**params_modelo)                                 # params del tuning Optuna — no cambia nada del modelo

scores_exp2  = []
n_trees_exp2 = []

print("\n" + "=" * 65)
print("  CV — LightGBM tuned con ccs_mean_acuity + temperature_valor")
print("=" * 65)

for fold, (idx_tr, idx_val) in enumerate(CV.split(X_tr_exp2, y_train, groups=groups_train)):

    X_fold_tr  = X_tr_exp2.iloc[idx_tr]
    X_fold_val = X_tr_exp2.iloc[idx_val]
    y_fold_tr  = y_train.iloc[idx_tr] - 1
    y_fold_val = y_train.iloc[idx_val] - 1
    sw_fold_tr  = sample_weights_train[idx_tr]
    sw_fold_val = sample_weights_train[idx_val]

    lgbm_exp2.fit(
        X_fold_tr, y_fold_tr,
        sample_weight      = sw_fold_tr,
        eval_set           = [(X_fold_val, y_fold_val)],
        eval_sample_weight = [sw_fold_val],
        callbacks          = [
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(period=-1),
        ],
    )

    y_pred = lgbm_exp2.predict(X_fold_val) + 1
    score  = macro_f1(y_train.iloc[idx_val], y_pred)
    scores_exp2.append(score)
    n_trees_exp2.append(lgbm_exp2.best_iteration_)
    print(f"  Fold {fold+1} — Macro F1: {score:.4f}  |  árboles: {lgbm_exp2.best_iteration_}")

f1_exp2  = np.mean(scores_exp2)
ganancia = f1_exp2 - f1_baseline

print(f"\n  LightGBM exp2 (48 features): {f1_exp2:.4f} ± {np.std(scores_exp2):.4f}")
print(f"  Baseline tuned (46 features): {f1_baseline:.4f}")
print(f"  Ganancia:                    {ganancia:+.4f}")
print(f"  Árboles promedio: {np.mean(n_trees_exp2):.0f}")

# ---------------------------------------------------------
# DECISIÓN
# ---------------------------------------------------------
UMBRAL = 0.003

print("\n" + "=" * 65)
if ganancia > UMBRAL:
    print(f"  ✅ Ganancia {ganancia:+.4f} > {UMBRAL} — incorporar al pipeline")

    X_tr_exp2.to_parquet(DATA_PROCESSED / "X_train_exp2.parquet", index=False)
    X_te_exp2.to_parquet(DATA_PROCESSED / "X_test_exp2.parquet",  index=False)

    config_exp2 = {
        "features_originales":  FEATURES_SELECCIONADAS,
        "features_nuevas":      ["ccs_mean_acuity", "temperature_valor"],
        "todas_features":       list(X_tr_exp2.columns),
        "n_features":           X_tr_exp2.shape[1],
        "macro_f1_baseline":    f1_baseline,
        "macro_f1_exp2":        round(f1_exp2, 6),
        "ganancia":             round(ganancia, 6),
        "ccs_encoding":         "target_mean_acuity_train_only",             # documentar que el encoding es solo sobre train
        "temperature_impute":   float(temp_mediana),                         # documentar el valor de imputación para reproducibilidad
    }
    with open(DATA_PROCESSED / "feature_config_exp2.json", "w", encoding="utf-8") as f:
        json.dump(config_exp2, f, indent=2, ensure_ascii=False)

    print(f"  Exportados: X_train_exp2.parquet, X_test_exp2.parquet, feature_config_exp2.json")
else:
    print(f"  ❌ Ganancia {ganancia:+.4f} ≤ {UMBRAL} — pipeline original se mantiene")
    print(f"  Ningún artefacto existente ha sido modificado ✅")
print("=" * 65)

X_tr_exp2: (334480, 48)  (46 originales + ccs_mean_acuity + temperature_valor)

  CV — LightGBM tuned con ccs_mean_acuity + temperature_valor
  Fold 1 — Macro F1: 0.5419  |  árboles: 530
  Fold 2 — Macro F1: 0.5456  |  árboles: 459


KeyboardInterrupt: 

In [5]:
# Solo temperature — ccs_mean_acuity eliminada
X_tr_exp3 = pd.concat([
    X_tr_sel.reset_index(drop=True),
    feats_temp_train,                    # solo esta
], axis=1)

X_te_exp3 = pd.concat([
    X_te_sel.reset_index(drop=True),
    feats_temp_test,
], axis=1)

print(f"X_tr_exp3: {X_tr_exp3.shape}  (46 originales + temperature_valor únicamente)")

lgbm_exp3    = LGBMClassifier(**params_modelo)
scores_exp3  = []
n_trees_exp3 = []

print("\n" + "=" * 65)
print("  CV — LightGBM tuned + temperature_valor (sin leakage)")
print("=" * 65)

for fold, (idx_tr, idx_val) in enumerate(CV.split(X_tr_exp3, y_train, groups=groups_train)):

    X_fold_tr  = X_tr_exp3.iloc[idx_tr]
    X_fold_val = X_tr_exp3.iloc[idx_val]
    y_fold_tr  = y_train.iloc[idx_tr] - 1
    y_fold_val = y_train.iloc[idx_val] - 1
    sw_fold_tr  = sample_weights_train[idx_tr]
    sw_fold_val = sample_weights_train[idx_val]

    lgbm_exp3.fit(
        X_fold_tr, y_fold_tr,
        sample_weight      = sw_fold_tr,
        eval_set           = [(X_fold_val, y_fold_val)],
        eval_sample_weight = [sw_fold_val],
        callbacks          = [
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(period=-1),
        ],
    )

    y_pred = lgbm_exp3.predict(X_fold_val) + 1
    score  = macro_f1(y_train.iloc[idx_val], y_pred)
    scores_exp3.append(score)
    n_trees_exp3.append(lgbm_exp3.best_iteration_)
    print(f"  Fold {fold+1} — Macro F1: {score:.4f}  |  árboles: {lgbm_exp3.best_iteration_}")

f1_exp3  = np.mean(scores_exp3)
ganancia = f1_exp3 - f1_baseline

print(f"\n  LightGBM + temperature (47 features): {f1_exp3:.4f} ± {np.std(scores_exp3):.4f}")
print(f"  Baseline tuned (46 features):          {f1_baseline:.4f}")
print(f"  Ganancia:                             {ganancia:+.4f}")

X_tr_exp3: (334480, 47)  (46 originales + temperature_valor únicamente)

  CV — LightGBM tuned + temperature_valor (sin leakage)
  Fold 1 — Macro F1: 0.4922  |  árboles: 435
  Fold 2 — Macro F1: 0.4806  |  árboles: 386
  Fold 3 — Macro F1: 0.5023  |  árboles: 446
  Fold 4 — Macro F1: 0.4882  |  árboles: 396
  Fold 5 — Macro F1: 0.4891  |  árboles: 409

  LightGBM + temperature (47 features): 0.4905 ± 0.0070
  Baseline tuned (46 features):          0.4871
  Ganancia:                             +0.0034


In [6]:
# =========================================================
# EXPERIMENTO 4 — temperature + heartrate + dbp
# =========================================================
# Las tres fueron descartadas en 05b por el flag de missing
# siendo más predictivo que el valor. Con num_leaves=132 y
# max_depth=9 el modelo ahora puede explotar el valor real.
# Testeamos las tres juntas y luego por separado si no mejora.
# =========================================================

# Medianas calculadas SOLO sobre train — sin leakage
hr_mediana  = df_train["heartrate"].median()
dbp_mediana = df_train["dbp"].median()

feats_hr_train = df_train["heartrate"].fillna(hr_mediana).rename("heartrate_valor").reset_index(drop=True)
feats_hr_test  = df_test["heartrate"].fillna(hr_mediana).rename("heartrate_valor").reset_index(drop=True)

feats_dbp_train = df_train["dbp"].fillna(dbp_mediana).rename("dbp_valor").reset_index(drop=True)
feats_dbp_test  = df_test["dbp"].fillna(dbp_mediana).rename("dbp_valor").reset_index(drop=True)

# Validación univariante rápida
from scipy.stats import kruskal

for nombre, feat in [("heartrate_valor", feats_hr_train), ("dbp_valor", feats_dbp_train)]:
    grupos = [feat[y_train.values == k].values for k in sorted(y_train.unique())]
    H, _   = kruskal(*grupos)
    nulos  = df_train["heartrate" if "heart" in nombre else "dbp"].isna().sum()
    print(f"{nombre:<20} H={H:>8.1f}  nulos_train={nulos:,} ({nulos/len(df_train)*100:.1f}%)")

print(f"temperature_valor      H=   405.6  (referencia — ya validada)")

# ---------------------------------------------------------
# X con las 3 features de valor vital
# ---------------------------------------------------------
X_tr_exp4 = pd.concat([
    X_tr_sel.reset_index(drop=True),
    feats_temp_train,                                        # ya calculada en exp3
    feats_hr_train,
    feats_dbp_train,
], axis=1)

X_te_exp4 = pd.concat([
    X_te_sel.reset_index(drop=True),
    feats_temp_test,
    feats_hr_test,
    feats_dbp_test,
], axis=1)

print(f"\nX_tr_exp4: {X_tr_exp4.shape}  (46 originales + temperature + heartrate + dbp)")

lgbm_exp4    = LGBMClassifier(**params_modelo)
scores_exp4  = []
n_trees_exp4 = []

print("\n" + "=" * 65)
print("  CV — temperature_valor + heartrate_valor + dbp_valor")
print("=" * 65)

for fold, (idx_tr, idx_val) in enumerate(CV.split(X_tr_exp4, y_train, groups=groups_train)):

    X_fold_tr  = X_tr_exp4.iloc[idx_tr]
    X_fold_val = X_tr_exp4.iloc[idx_val]
    y_fold_tr  = y_train.iloc[idx_tr] - 1
    y_fold_val = y_train.iloc[idx_val] - 1
    sw_fold_tr  = sample_weights_train[idx_tr]
    sw_fold_val = sample_weights_train[idx_val]

    lgbm_exp4.fit(
        X_fold_tr, y_fold_tr,
        sample_weight      = sw_fold_tr,
        eval_set           = [(X_fold_val, y_fold_val)],
        eval_sample_weight = [sw_fold_val],
        callbacks          = [
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(period=-1),
        ],
    )

    y_pred = lgbm_exp4.predict(X_fold_val) + 1
    score  = macro_f1(y_train.iloc[idx_val], y_pred)
    scores_exp4.append(score)
    n_trees_exp4.append(lgbm_exp4.best_iteration_)
    print(f"  Fold {fold+1} — Macro F1: {score:.4f}  |  árboles: {lgbm_exp4.best_iteration_}")

f1_exp4   = np.mean(scores_exp4)
f1_exp3   = 0.4905                                                           # referencia: solo temperature
ganancia4 = f1_exp4 - f1_baseline

print(f"\n  LightGBM + 3 vitales (49 features): {f1_exp4:.4f} ± {np.std(scores_exp4):.4f}")
print(f"  LightGBM + temperature solo:         {f1_exp3:.4f}")
print(f"  Baseline tuned (46 features):        {f1_baseline:.4f}")
print(f"  Ganancia vs baseline:               {ganancia4:+.4f}")
print(f"  Ganancia vs solo temperature:       {f1_exp4 - f1_exp3:+.4f}")
print(f"  Árboles promedio: {np.mean(n_trees_exp4):.0f}")

# ---------------------------------------------------------
# DECISIÓN
# ---------------------------------------------------------
print("\n" + "=" * 65)
UMBRAL = 0.003

if ganancia4 > UMBRAL:
    print(f"  ✅ Ganancia {ganancia4:+.4f} — las 3 features juntas mejoran el pipeline")

    X_tr_exp4.to_parquet(DATA_PROCESSED / "X_train_exp4.parquet", index=False)
    X_te_exp4.to_parquet(DATA_PROCESSED / "X_test_exp4.parquet",  index=False)

    config_exp4 = {
        "features_originales":   FEATURES_SELECCIONADAS,
        "features_nuevas":       ["temperature_valor", "heartrate_valor", "dbp_valor"],
        "todas_features":        list(X_tr_exp4.columns),
        "n_features":            X_tr_exp4.shape[1],
        "macro_f1_baseline":     f1_baseline,
        "macro_f1_exp3_temp":    f1_exp3,
        "macro_f1_exp4":         round(f1_exp4, 6),
        "ganancia_vs_baseline":  round(ganancia4, 6),
        "imputaciones": {
            "temperature": float(temp_mediana),
            "heartrate":   float(hr_mediana),
            "dbp":         float(dbp_mediana),
        }
    }
    with open(DATA_PROCESSED / "feature_config_exp4.json", "w", encoding="utf-8") as f:
        json.dump(config_exp4, f, indent=2, ensure_ascii=False)

    print(f"  Exportados: X_train_exp4.parquet, X_test_exp4.parquet, feature_config_exp4.json")

elif f1_exp4 > f1_exp3:
    print(f"  ⚠️  Ganancia {ganancia4:+.4f} — mejora vs baseline pero marginal")
    print(f"  → heartrate y dbp añaden algo sobre temperature sola")
    print(f"  → considera usar solo temperature (exp3) como pipeline definitivo")

else:
    print(f"  ❌ Ganancia {ganancia4:+.4f} — heartrate y dbp no aportan sobre temperature")
    print(f"  → Pipeline definitivo: 47 features con solo temperature_valor (exp3)")

print("=" * 65)

heartrate_valor      H=  2339.7  nulos_train=8,495 (2.5%)
dbp_valor            H=   847.1  nulos_train=10,499 (3.1%)
temperature_valor      H=   405.6  (referencia — ya validada)

X_tr_exp4: (334480, 49)  (46 originales + temperature + heartrate + dbp)

  CV — temperature_valor + heartrate_valor + dbp_valor
  Fold 1 — Macro F1: 0.4944  |  árboles: 408
  Fold 2 — Macro F1: 0.4847  |  árboles: 386
  Fold 3 — Macro F1: 0.5019  |  árboles: 436
  Fold 4 — Macro F1: 0.4926  |  árboles: 381
  Fold 5 — Macro F1: 0.4918  |  árboles: 411

  LightGBM + 3 vitales (49 features): 0.4931 ± 0.0055
  LightGBM + temperature solo:         0.4905
  Baseline tuned (46 features):        0.4871
  Ganancia vs baseline:               +0.0060
  Ganancia vs solo temperature:       +0.0026
  Árboles promedio: 404

  ✅ Ganancia +0.0060 — las 3 features juntas mejoran el pipeline
  Exportados: X_train_exp4.parquet, X_test_exp4.parquet, feature_config_exp4.json


In [ ]:
# =========================================================
# EXPERIMENTO 5 — Kitchen sink con las 88 features completas
# =========================================================

# ---------------------------------------------------------
# 1. CARGA DEL DATASET COMPLETO (88 features)
# ---------------------------------------------------------
dataset_completo = pd.read_parquet(DATA_PROCESSED / "dataset_features.parquet")

fecha_corte = df_original["intime"].quantile(0.80)                          # mismo corte temporal que el pipeline original
mask_train  = df_original["intime"] <= fecha_corte
mask_test   = df_original["intime"] >  fecha_corte

df_88_train = dataset_completo[mask_train.values].reset_index(drop=True)   # alineado con X_train por orden temporal
df_88_test  = dataset_completo[mask_test.values].reset_index(drop=True)

assert len(df_88_train) == len(X_train), \
    f"Desalineación train: {len(df_88_train)} vs {len(X_train)}"
assert len(df_88_test) == len(X_test), \
    f"Desalineación test: {len(df_88_test)} vs {len(X_test)}"

print(f"Dataset completo cargado: {dataset_completo.shape}")
print(f"df_88_train: {df_88_train.shape} ✅")
print(f"df_88_test:  {df_88_test.shape}  ✅")

# Features descartadas en 05b = todas_88 - seleccionadas - target
DESCARTADAS_88 = [f for f in dataset_completo.columns
                  if f not in FEATURES_SELECCIONADAS
                  and f != TARGET]

print(f"\nFeatures descartadas recuperables: {len(DESCARTADAS_88)}")

# ---------------------------------------------------------
# 2. GRUPOS CLÍNICOS DE FEATURES DESCARTADAS
# ---------------------------------------------------------
GRUPOS_DESCARTADAS = {

    "vitales_flags": [                                                        # flags derivados de vitales — descartados por correlación con continuas
        "fiebre", "hipotension", "hipertension_severa",
        "qsofa_positivo", "shock_index_severo",
        "anciano", "anciano_mayor",
    ],

    "visitas": [                                                              # historial de visitas — baja eta2 individual con acuity
        "n_visitas_previas", "visitas_ultimo_mes",
        "visitas_ultimo_año", "dias_desde_ultima_visita",
        "primera_visita", "frecuentador",
    ],

    "cc_extra": [                                                             # CC flags con baja prevalencia o Cramér V < umbral
        "cc_cefalea", "cc_intoxicacion",
        "cc_infeccioso", "cc_hemorragia_activa",
    ],

    "hx_extra": [                                                             # antecedentes con baja asociación individual con acuity
        "hx_respiratorio", "hx_neuro", "hx_psiquiatrico",
        "hx_abuso_sustancias", "hx_digestivo",
        "hx_metabolico_renal", "hx_infeccioso", "hx_trauma_muscular",
    ],

    "med_extra": [                                                            # medicamentos con Cramér V < 0.07 individualmente
        "med_anticoagulante", "med_antidiabetico_oral",
        "med_corticoide_sistemico", "med_benzodiacepina",
        "med_diuretico_tiazida", "med_respiratorio_inhalado",
        "med_inmunosupresor", "med_digoxina", "med_antiaritmico",
        "riesgo_depresion_resp", "polifarmacia", "hiperpolifarmacia",
    ],

    "llegada_extra": [                                                        # llegada_autonoma: complemento de llegada_ambulancia
        "llegada_autonoma",
    ],
}

# Verificar disponibilidad en df_88_train
for grupo, features in GRUPOS_DESCARTADAS.items():
    disponibles = [f for f in features if f in df_88_train.columns]
    faltantes   = [f for f in features if f not in df_88_train.columns]
    print(f"  {grupo:<20} {len(disponibles)}/{len(features)} disponibles"
          + (f" | faltantes: {faltantes}" if faltantes else ""))

TODAS_DESCARTADAS = [f for grupo in GRUPOS_DESCARTADAS.values()
                     for f in grupo if f in df_88_train.columns]
print(f"\nTotal features a testear: {len(TODAS_DESCARTADAS)}")

# ---------------------------------------------------------
# 3. VALIDACIÓN UNIVARIANTE — H Kruskal por grupo
# ---------------------------------------------------------
print("\n" + "=" * 65)
print("  VALIDACIÓN UNIVARIANTE POR GRUPO")
print("=" * 65)

features_con_senal = []

for grupo, features in GRUPOS_DESCARTADAS.items():
    feats_grupo = [f for f in features if f in df_88_train.columns]
    if not feats_grupo:
        continue
    print(f"\n  [{grupo}]")
    for f in feats_grupo:
        col = df_88_train[f].fillna(0)
        if col.std() > 0:
            grupos_k = [col[y_train.values == k].values
                        for k in sorted(y_train.unique())]
            H, _ = kruskal(*grupos_k)
            flag = "✅" if H > 50 else ("⚠️ " if H > 10 else "🚨")
            print(f"    {flag} {f:<35} H={H:>8.1f}")
            if H > 10:
                features_con_senal.append(f)
        else:
            print(f"    🚨 {f:<35} sin varianza")

print(f"\n  Features con señal (H>10): {len(features_con_senal)}/{len(TODAS_DESCARTADAS)}")
print(f"  {features_con_senal}")

# ---------------------------------------------------------
# 4. EXPERIMENTO 5A — kitchen sink sobre exp4
# ---------------------------------------------------------
feats_ks_train = df_88_train[features_con_senal].fillna(0).reset_index(drop=True).astype("float32")
feats_ks_test  = df_88_test[features_con_senal].fillna(0).reset_index(drop=True).astype("float32")

X_tr_exp5 = pd.concat([
    X_tr_exp4.reset_index(drop=True),                                        # base: 49 features del exp4
    feats_ks_train,
], axis=1)

X_te_exp5 = pd.concat([
    X_te_exp4.reset_index(drop=True),
    feats_ks_test,
], axis=1)

print(f"\nX_tr_exp5: {X_tr_exp5.shape}  "
      f"(49 exp4 + {len(features_con_senal)} descartadas con señal)")

lgbm_exp5    = LGBMClassifier(**params_modelo)
scores_exp5  = []
n_trees_exp5 = []

print("\n" + "=" * 65)
print(f"  CV — Kitchen sink: exp4 + {len(features_con_senal)} features descartadas")
print("=" * 65)

for fold, (idx_tr, idx_val) in enumerate(CV.split(X_tr_exp5, y_train, groups=groups_train)):

    X_fold_tr  = X_tr_exp5.iloc[idx_tr]
    X_fold_val = X_tr_exp5.iloc[idx_val]
    y_fold_tr  = y_train.iloc[idx_tr] - 1
    y_fold_val = y_train.iloc[idx_val] - 1
    sw_fold_tr  = sample_weights_train[idx_tr]
    sw_fold_val = sample_weights_train[idx_val]

    lgbm_exp5.fit(
        X_fold_tr, y_fold_tr,
        sample_weight      = sw_fold_tr,
        eval_set           = [(X_fold_val, y_fold_val)],
        eval_sample_weight = [sw_fold_val],
        callbacks          = [
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(period=-1),
        ],
    )

    y_pred = lgbm_exp5.predict(X_fold_val) + 1
    score  = macro_f1(y_train.iloc[idx_val], y_pred)
    scores_exp5.append(score)
    n_trees_exp5.append(lgbm_exp5.best_iteration_)
    print(f"  Fold {fold+1} — Macro F1: {score:.4f}  |  árboles: {lgbm_exp5.best_iteration_}")

f1_exp5    = np.mean(scores_exp5)
f1_exp4    = 0.4931
ganancia5  = f1_exp5 - f1_baseline
delta_exp4 = f1_exp5 - f1_exp4

print(f"\n  Kitchen sink ({X_tr_exp5.shape[1]} features): {f1_exp5:.4f} ± {np.std(scores_exp5):.4f}")
print(f"  Exp4 (49 features):                       {f1_exp4:.4f}")
print(f"  Baseline (46 features):                   {f1_baseline:.4f}")
print(f"  Ganancia vs baseline:                    {ganancia5:+.4f}")
print(f"  Ganancia vs exp4:                        {delta_exp4:+.4f}")
print(f"  Árboles promedio: {np.mean(n_trees_exp5):.0f}")

# ---------------------------------------------------------
# 5. ABLACIÓN POR GRUPO — solo si kitchen sink mejora exp4
# ---------------------------------------------------------
if delta_exp4 > 0.001:
    print(f"\n  ✅ Kitchen sink mejora exp4 → ablación por grupo")
    print("\n" + "=" * 65)
    print("  ABLACIÓN — añadiendo cada grupo individualmente sobre exp4")
    print("=" * 65)

    resultados_ablacion = {}

    for grupo, features in GRUPOS_DESCARTADAS.items():
        feats_grupo = [f for f in features if f in features_con_senal]
        if not feats_grupo:
            print(f"  {'⬛'} {grupo:<20} — sin features con señal")
            continue

        X_tr_abl = pd.concat([
            X_tr_exp4.reset_index(drop=True),
            df_88_train[feats_grupo].fillna(0).reset_index(drop=True).astype("float32"),
        ], axis=1)

        scores_abl = []
        lgbm_abl   = LGBMClassifier(**params_modelo)

        for fold, (idx_tr, idx_val) in enumerate(CV.split(X_tr_abl, y_train, groups=groups_train)):
            lgbm_abl.fit(
                X_tr_abl.iloc[idx_tr], y_train.iloc[idx_tr] - 1,
                sample_weight      = sample_weights_train[idx_tr],
                eval_set           = [(X_tr_abl.iloc[idx_val],
                                       y_train.iloc[idx_val] - 1)],
                eval_sample_weight = [sample_weights_train[idx_val]],
                callbacks          = [
                    lgb.early_stopping(50, verbose=False),
                    lgb.log_evaluation(period=-1),
                ],
            )
            scores_abl.append(macro_f1(
                y_train.iloc[idx_val],
                lgbm_abl.predict(X_tr_abl.iloc[idx_val]) + 1
            ))

        f1_abl = np.mean(scores_abl)
        delta  = f1_abl - f1_exp4
        flag   = "✅" if delta > 0.001 else ("⚠️ " if delta > 0 else "❌")
        resultados_ablacion[grupo] = {"f1": f1_abl, "delta": delta, "features": feats_grupo}
        print(f"  {flag} {grupo:<20} F1={f1_abl:.4f}  Δexp4={delta:+.4f}  n={len(feats_grupo)}")

    print(f"\n  Ranking de grupos:")
    for g, v in sorted(resultados_ablacion.items(), key=lambda x: -x[1]["f1"]):
        print(f"    {g:<20} {v['f1']:.4f}  {v['delta']:+.4f}  {v['features']}")

    # Exportar si hay ganancia real
    mejor_grupo = max(resultados_ablacion, key=lambda x: resultados_ablacion[x]["f1"])
    if resultados_ablacion[mejor_grupo]["delta"] > 0.001:
        print(f"\n  Mejor grupo: {mejor_grupo} → incorporar al pipeline")
else:
    print(f"\n  ❌ Kitchen sink no mejora exp4 ({delta_exp4:+.4f})")
    print(f"  → Pipeline definitivo: exp4 con 49 features (Macro F1: {f1_exp4:.4f})")
    print(f"  → Ningún artefacto modificado ✅")

print("=" * 65)

Dataset completo cargado: (418100, 89)
df_88_train: (334480, 89) ✅
df_88_test:  (83620, 89)  ✅

Features descartadas recuperables: 42
  vitales_flags        7/7 disponibles
  visitas              6/6 disponibles
  cc_extra             4/4 disponibles
  hx_extra             8/8 disponibles
  med_extra            12/12 disponibles
  llegada_extra        1/1 disponibles

Total features a testear: 38

  VALIDACIÓN UNIVARIANTE POR GRUPO

  [vitales_flags]
    ✅ fiebre                              H=   815.3
    ✅ hipotension                         H= 24461.6
    ✅ hipertension_severa                 H=   883.6
    ✅ qsofa_positivo                      H=  2122.4
    ✅ shock_index_severo                  H=  8137.0
    ✅ anciano                             H= 13476.7
    ✅ anciano_mayor                       H=  9004.9

  [visitas]
    ✅ n_visitas_previas                   H=   729.8
    ✅ visitas_ultimo_mes                  H=   389.3
    ✅ visitas_ultimo_año                  H=   719.6
  